In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


In [9]:
AWS_ACCESS_KEY_ID = "xxxxxxxxx"
AWS_SECRET_ACCESS_KEY = "xxxxxxxxxx"
AWS_REGION = "eu-central-1"

INPUT_PATH = "s3a://cde-spark-demo/demo/orders/"
OUTPUT_PATH = "s3a://cde-spark-demo/demo/revenue_by_region/"


In [10]:
spark = (
    SparkSession.builder.appName("RevenueByRegion")
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID)
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.endpoint.region", AWS_REGION)
    .getOrCreate()
)

In [11]:
orders = spark.read.parquet(INPUT_PATH)

paid_orders = orders.filter(F.col("status") == "PAID")

# revenue_by_region = (
#     paid_orders.groupBy("region")
#     .agg(F.sum("amount").alias("total_revenue"))
#     .orderBy("region")
# )

paid_orders.show()


+--------+------+------+------+
|order_id|region|amount|status|
+--------+------+------+------+
|     101|    EU|   200|  PAID|
|     103|    EU|   300|  PAID|
|     104|    US|   150|  PAID|
|     105|    EU|   400|  PAID|
+--------+------+------+------+



In [12]:
revenue_by_region = (
    paid_orders.groupBy("region")
    .agg(F.sum("amount").alias("total_revenue"))
    .orderBy("region")
)

revenue_by_region.show()

+------+-------------+
|region|total_revenue|
+------+-------------+
|    EU|          900|
|    US|          150|
+------+-------------+



In [13]:
revenue_by_region.coalesce(1).write.mode("overwrite").parquet(OUTPUT_PATH)

In [14]:
spark.stop()